# Candidate Frame Selection - Feature Analysis & Machine Learning Pipeline

This notebook provides a complete pipeline to:
1. **Load and clean** the generated pairwise feature dataset (dealing with NaNs, infinites, and extreme Chi-Square outliers).
2. **Explore and visualize** feature distributions, target label correlation, and class separability.
3. **Train baseline machine learning models** (Logistic Regression and Random Forest) to classify whether a frame pair is a candidate transition (Keep vs Discard).
4. **Evaluate model performance** using ROC curves, confusion matrices, and feature importances.

In [1]:
# Install dependencies if running on Google Colab
# !pip install -q seaborn matplotlib pandas numpy scikit-learn

import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

# Plot aesthetics setup for dark/light themes
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

## 1. Load Dataset

We detect if the environment is Google Colab or local to load the dataset automatically.
* Data Path: `generated_datasets/session_v3_crops/datasets/candidate_frame_dataset_v3_crops.csv`

In [4]:
local_path = "generated_datasets/session_v3_crops/datasets/candidate_frame_dataset_v3_crops.csv"
df = pd.read_csv(local_path)
print("\nDataset loaded successfully.")
print("Dataset Shape:", df.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'generated_datasets/session_v3_crops/datasets/candidate_frame_dataset_v3_crops.csv'

## 2. Data Cleaning & Outlier Mitigation

Histogram metrics (specifically Chi-Square distance) can sometimes explode to extremely large numbers ($10^{18}$) or produce infinite values due to near-zero denominators.
We will:
1. Check and fill any missing values (NaN).
2. Replace `inf` / `-inf` values with NaN, then impute them with feature medians.
3. Clip extreme outlier values exceeding 10x the 99th percentile of finite values to stabilize model training.
4. Remove constant (zero-variance) features.

In [ ]:
# 1. Check for missing values
missing = df.isnull().sum().sum()
print(f"Initial missing values count: {missing}")
if missing > 0:
    df = df.fillna(df.median(numeric_only=True))

# 2. Check for infinite values (inf)
inf_count = np.isinf(df.select_dtypes(include=np.number)).sum().sum()
print(f"Infinite values count: {inf_count}")
if inf_count > 0:
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.fillna(df.median(numeric_only=True))

# 3. Outlier Clipping (especially for Chi-Square distance columns)
numeric_cols = df.select_dtypes(include=np.number).columns.drop('GroundTruth', errors='ignore')
clipped_count = 0
for col in numeric_cols:
    q99 = df[col].quantile(0.99)
    if pd.notnull(q99) and q99 > 0:
        is_outlier = df[col] > (10 * q99)
        if is_outlier.any():
            df[col] = df[col].clip(upper=10 * q99)
            clipped_count += is_outlier.sum()
print(f"Clipped {clipped_count} extreme outlier values.")

# 4. Remove constant (zero-variance) columns
constant_cols = [col for col in numeric_cols if df[col].std() == 0]
if constant_cols:
    df = df.drop(columns=constant_cols)
    print(f"Dropped {len(constant_cols)} constant columns: {constant_cols}")

print(f"\nFinal cleaned dataset shape: {df.shape}")

## 3. Exploratory Data Analysis (EDA) & Visualization

Let's explore the distribution of the target class (`GroundTruth`) and visualize the relationship of key features to class separation.

In [ ]:
# Target Label Distribution
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='GroundTruth', hue='GroundTruth', legend=False)
plt.title("Target Label Distribution (GroundTruth)")
plt.xlabel("Label (0: Discard, 1: Keep)")
plt.ylabel("Sample Count")
plt.show()

counts = df['GroundTruth'].value_counts()
percentages = df['GroundTruth'].value_counts(normalize=True) * 100
print("Class Distribution:")
for val, count in counts.items():
    print(f"  Class {val}: {count} samples ({percentages[val]:.2f}%)")

### Feature Correlations with GroundTruth
Let's identify and plot the features that are most strongly correlated (both positively and negatively) with the transition target class.

In [ ]:
# Compute Pearson correlation coefficients
correlations = df.corr(numeric_only=True)['GroundTruth'].drop('GroundTruth', errors='ignore').sort_values()

# Select top 10 negative and top 10 positive correlations
top_neg = correlations.head(10)
top_pos = correlations.tail(10)
top_corr = pd.concat([top_neg, top_pos])

plt.figure(figsize=(10, 8))
sns.barplot(x=top_corr.values, y=top_corr.index, hue=top_corr.index, legend=False, palette="coolwarm")
plt.title("Top 20 Features Correlated with GroundTruth Label")
plt.xlabel("Correlation Coefficient")
plt.ylabel("Features")
plt.axvline(0, color='black', linewidth=1, linestyle='--')
plt.show()

### Key Feature Distributions
We visualize key features like `SSIM_Mean` (structural difference), `Mean_Absolute_Difference` (average pixel intensity difference), `Text_Occupancy_Diff` (text structure changes), and `Whole_Edge_Density_Diff` (edge complexity differences) across both target classes.

In [ ]:
key_cols = ['SSIM_Mean', 'Mean_Absolute_Difference', 'Text_Occupancy_Diff', 'Whole_Edge_Density_Diff']
existing_keys = [col for col in key_cols if col in df.columns]

if existing_keys:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()
    for i, col in enumerate(existing_keys):
        sns.kdeplot(data=df, x=col, hue='GroundTruth', fill=True, common_norm=False, alpha=0.5, ax=axes[i])
        axes[i].set_title(f"Distribution of {col} by Class")
        axes[i].set_xlabel(col)
    plt.tight_layout()
    plt.show()

### Bivariate Distribution (SSIM vs MAD)
A scatter plot comparing SSIM (structural) and Mean Absolute Difference (color/intensity shift) to show how they partition the Keep vs Discard classes.

In [ ]:
if 'SSIM_Mean' in df.columns and 'Mean_Absolute_Difference' in df.columns:
    plt.figure(figsize=(8, 6))
    sns.scatterplot(data=df, x='SSIM_Mean', y='Mean_Absolute_Difference', hue='GroundTruth', alpha=0.7, style='GroundTruth')
    plt.title("Bivariate Separation: SSIM_Mean vs Mean_Absolute_Difference (MAD)")
    plt.xlabel("SSIM Mean")
    plt.ylabel("Mean Absolute Difference (MAD)")
    plt.show()

## 4. Machine Learning Model Training & Evaluation

We split the cleaned dataset into a training (80%) and testing (20%) set, apply a robust scaling preprocessing step, and train two classification models:
1. **Logistic Regression** (linear classification baseline)
2. **Random Forest Classifier** (non-linear tree-based classifier)

In [ ]:
# 1. Split features and target label
X = df.drop(columns=['GroundTruth'])
y = df['GroundTruth']

# 2. Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training subset shape: {X_train.shape}")
print(f"Testing subset shape: {X_test.shape}")

# 3. Scaling using RobustScaler (resilient to residual outliers)
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Train Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)
y_pred_lr = lr_model.predict(X_test_scaled)
y_prob_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

# 5. Train Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)
y_pred_rf = rf_model.predict(X_test_scaled)
y_prob_rf = rf_model.predict_proba(X_test_scaled)[:, 1]

## 5. Model Evaluation Results

In [ ]:
print("==============================================")
print("     LOGISTIC REGRESSION CLASSIFICATION REPORT")
print("==============================================")
print(classification_report(y_test, y_pred_lr, target_names=["Discard (0)", "Keep (1)"]))

print("\n==============================================")
print("       RANDOM FOREST CLASSIFICATION REPORT")
print("==============================================")
print(classification_report(y_test, y_pred_rf, target_names=["Discard (0)", "Keep (1)"]))

### ROC Curve and Confusion Matrix
We visualize the Receiver Operating Characteristic (ROC) curve to compare classification thresholds, and plot the confusion matrix for the Random Forest model.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 1. ROC Curves
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
auc_lr = roc_auc_score(y_test, y_prob_lr)
auc_rf = roc_auc_score(y_test, y_prob_rf)

axes[0].plot(fpr_lr, tpr_lr, label=f"Logistic Regression (AUC = {auc_lr:.4f})", lw=2)
axes[0].plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC = {auc_rf:.4f})", lw=2)
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label="Random Guess")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curves Comparison")
axes[0].legend(loc="lower right")

# 2. Confusion Matrix
cm = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1], cbar=False)
axes[1].set_xlabel("Predicted Label")
axes[1].set_ylabel("True Label")
axes[1].set_title("Random Forest Confusion Matrix")
axes[1].set_xticklabels(['Discard (0)', 'Keep (1)'])
axes[1].set_yticklabels(['Discard (0)', 'Keep (1)'])

plt.tight_layout()
plt.show()

### Feature Importance
Let's see which features contribute the most to the Random Forest model's predictions.

In [ ]:
# Compute importances
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

# Top 15 features
top_k = 15
top_importances = importances[indices[:top_k]]
top_features = X.columns[indices[:top_k]]

plt.figure(figsize=(10, 6))
sns.barplot(x=top_importances, y=top_features, hue=top_features, legend=False, palette="viridis")
plt.title(f"Top {top_k} Feature Importances (Random Forest)")
plt.xlabel("Relative Importance (Mean Decrease in Impurity)")
plt.ylabel("Features")
plt.show()

print("=== TOP 15 FEATURES BY IMPORTANCE ===")
for rank, idx in enumerate(indices[:top_k]):
    print(f"Rank {rank+1:2d}: {X.columns[idx]:50s} ({importances[idx]:.4f})")